In [ ]:
import os

import torch
from torch import nn
import torchvision
from torch.utils.data import DataLoader

from spbu_bachelor_thesis.datasets import CUB200Dataset, extract_embeddings
from spbu_bachelor_thesis.global_constants import DATA_DIR

# The random seed is fixed to ensure reproducibility of results
torch.manual_seed(42)
device = (
    torch.accelerator.current_accelerator()
    if torch.accelerator.is_available()
    else torch.device("cpu")
)
device

device(type='cuda')

In [2]:
# Create a directory for datasets if it doesn't exist
os.makedirs(DATA_DIR, exist_ok=True)

#### CUB200-2011 dataset


In [3]:
# These transformations preprocess the input images to match the format and statistics
# expected by models pre-trained on ImageNet, such as the ResNet-50
transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
    ]
)

# The dataset is downloaded if neccessary
trainset = CUB200Dataset(train=True, transform=transform)
testset = CUB200Dataset(train=False, transform=transform)
trainloader = DataLoader(trainset, batch_size=32, num_workers=4)
testloader = DataLoader(testset, batch_size=32, num_workers=4)

100%|██████████| 1.15G/1.15G [01:07<00:00, 17.1MB/s] 


In [4]:
# Download a pretrained ResNet50 model and remove the classifier (last fully
# connected layer) to generate image embeddings for the CUB200 dataset
resnet50 = torchvision.models.resnet50(
    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2
)
resnet50.fc = nn.Identity()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 94.5MB/s]


In [5]:
# Extract output of the ResNet-50's last hidden layer (embeddings) and save them
extract_embeddings(trainloader, resnet50, "cub200_train_embed.pt", device)
extract_embeddings(testloader, resnet50, "cub200_test_embed.pt", device)

Extracting embeddings to /root/spbu-bachelor-thesis/src/data/cub200_train_embed.pt: 100%|██████████| 188/188 [00:17<00:00, 10.67it/s]
Extracting embeddings to /root/spbu-bachelor-thesis/src/data/cub200_test_embed.pt: 100%|██████████| 182/182 [00:16<00:00, 10.89it/s]
